# 1단계 인코더 미세조정: 라벨 순서 · 태그 유사도

지금까지 GraphSAGE 구조·하이퍼파라미터·태그 입력/엣지를 바꿔도 정확도가 50%대 중반에서 멈췄다. 원인은 **고정된 GTE 임베딩이 라벨 기준(재료는 달라도 조리법·완성 형태가 비슷한가)과 맞지 않는 것**으로 보인다. 학습 쌍 코사인의 1점↔2점, 2점↔3점 순서 일치율이 0.57/0.63으로 동전 던지기에 가깝다. 이 노트북은 GTE 자체를 미세조정해서, 미세조정한 임베딩의 **코사인만으로** 라벨 순서가 얼마나 좋아지는지 본다.

**손실: CoSENT (순서만 맞춤)**
한 배치 안에서 목표값이 i > j인 두 쌍이 있으면 cos_i > cos_j가 되도록 `log(1 + Σ exp(s·(cos_j − cos_i)))`를 최소화한다. 코사인의 절댓값을 라벨/4에 맞추지 않으므로 사전학습 공간을 덜 망가뜨린다. 0~4점 변환은 GraphSAGE나 경계 조정에서 한다.

| 조건 | 라벨 손실 | 태그 손실 | 태그 쌍 풀 |
|---|---|---|---|
| `frozen` | – | – | – (미세조정 없음, 같은 입력의 기준선) |
| `label` | 1 | – | – |
| `tag` | – | 1 | 라벨링 레시피 2,914개 |
| `label+tag0.2` | 1 | 0.2 | 라벨링 레시피 2,914개 |
| `tag+extra` | – | 1 | + RAW에서 뽑은 라벨 없는 레시피 2만 개 |
| `label+tag0.2+extra` | 1 | 0.2 | + RAW에서 뽑은 라벨 없는 레시피 2만 개 |

- **라벨 배치:** 라벨은 기준 레시피 하나에 여러 레시피를 붙여 매긴 구조라서, 같은 기준 레시피의 쌍을 한 배치에 묶는다(이 레시피 기준 b1이 b2보다 비슷한가).
- **태그 목표:** 재료·영양·공통 분류 태그를 뺀 태그(완성 형태, 조리법·도구, 시간·난이도, 국적, 행사)의 **IDF 가중 Jaccard**. 기준 레시피마다 태그가 가까운 레시피 2개 + 무작위 2개를 짝지어 목표값이 고루 퍼지게 한다. 목표값 차이가 `tag_margin`(0.05)보다 작은 쌍끼리는 비교하지 않는다. 검증·테스트 라벨 쌍은 태그 쌍에서도 뺀다.
- **태그 Jaccard의 한계:** 라벨과의 Spearman 0.45로 GTE 코사인(0.62)보다 약하고, 1↔2·2↔3점 순서 일치율(0.59~0.62)도 GTE와 비슷하다. 그래서 태그 손실은 주 목표가 아니라 보조·정규화 역할로 두고, 단독 조건으로 효과를 따로 확인한다.
- **입력(`VIEW`):** 기본은 재료를 가린 `masked`다. 마스크 토큰은 기본적으로 `[MASK]`로 통일해 카테고리(`vegetables` 등)도 지우고, 레시피 이름 속 재료 이름도 가린다. RAW에서 가져온 추가 레시피는 마스킹 CSV가 없어 재료 리스트로 문자열을 찾아 가린다(근사).
- **평가:** 미세조정한 인코더로 모든 레시피를 기존과 같은 방식(512토큰 창 가중 평균)으로 인코딩하고 쌍 코사인을 본다.
  - 모델 선택(조기 종료)은 **검증 Spearman**으로 한다. 1↔2·2↔3점 순서 일치율도 같이 본다.
  - 정확도는 코사인을 0~4점으로 나누는 경계를 **검증에서 맞춰** 테스트에 적용한다. 검증 정확도는 낙관적인 값이다.
  - 학습 쌍 Spearman은 과적합 정도를 보는 용도다.
- **결과:** `results/experiments/05_encoder_finetune/<조건>/seed<시드>/`에 `metrics.json`, `history.json`, 그리고 **기존 1단계 pt와 같은 형식의 임베딩** `stage1_embeddings.pt`를 저장한다. 이 pt를 `stage1_graph_experiments.ipynb`의 `DATA`로 바로 쓸 수 있다(맨 아래 참고).

**준비 (Colab GPU 런타임 권장):** 아래 파일을 `/content/data/` 아래 같은 경로로 올린다. CPU에서는 인코딩 한 번에 1시간 이상 걸린다.

| 파일 | 용도 |
|---|---|
| `data/stage1_rebuild_raw_gte.pt` | 레시피 순서, train/val/test 분할 |
| `data/recipes_raw_v3.csv` | 레시피 이름·재료·원본 조리 단계 |
| `data/stage1/v3/recipes_masked_v3.csv` | 재료 마스킹 조리 단계 (`VIEW="masked"`) |
| `data/labeled_pairs_masked_v3.csv` | 라벨링 레시피의 태그 |
| `data/RAW_recipes_normalized.csv` (약 290MB) | `+extra` 조건만. 없으면 그 조건은 건너뜀 |

**시간 (T4 기준 대략):** 조건당 epoch마다 학습 2~3분 + 평가 인코딩 1~2분, 최대 6 epoch. 조건 6개 × 시드 1개에 2~3시간 정도 걸린다. 결과가 있는 조건은 다시 학습하지 않으므로 런타임이 끊기면 셀을 다시 실행하면 된다(끊기기 전에 마지막 셀로 내려받기).

In [ ]:
# ======================================================================
# [1] 설정
# ======================================================================
# Colab에 sentence-transformers가 없으면: !pip install -q sentence-transformers
from __future__ import annotations

import ast
import csv
import json
import math
import random
import re
import statistics
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import torch
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from torch.nn import functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "thenlper/gte-base"

DATA = Path("data/stage1_rebuild_raw_gte.pt")  # 레시피 순서 + train/val/test 분할 (기존 1단계와 동일)
RAW_RECIPES = Path("data/recipes_raw_v3.csv")  # 레시피 이름·재료·원본 조리 단계
MASKED_RECIPES = Path("data/stage1/v3/recipes_masked_v3.csv")  # 재료 마스킹 조리 단계
PAIRS_CSV = Path("data/labeled_pairs_masked_v3.csv")  # 라벨링 레시피의 태그
RAW_ALL_CANDIDATES = [Path("data/RAW_recipes_normalized.csv"), Path("../RAW_recipes_normalized.csv")]
RESULTS = Path("results/experiments/05_encoder_finetune")

VIEW = "masked"  # "masked" (재료 이름을 가림, 목표에 맞음) / "raw" (기존 1단계 입력)
MASK_STYLE = "generic"  # 마스크 토큰: "generic"=[MASK] / "category"=[MASK:카테고리] / "keep"=원래 토큰 그대로
MASK_NAME = True  # 레시피 이름 속 재료 이름도 가림 (예: "winter squash soup" -> "[MASK] soup")
SEEDS = [42]  # 미세조정은 비싸서 기본 1개. 여유가 있으면 [42, 43, 44]로 시드 노이즈 확인
EVAL_LIMIT = 512  # 평가 인코딩 창 크기 (기존 1단계 인코딩과 같음)
SMOKE_TEST = False  # True면 아주 작은 설정으로 코드 경로만 확인 (CPU에서도 몇 분)

if SMOKE_TEST:
    RESULTS = RESULTS / "smoke"
    EVAL_LIMIT = 64
print("torch", torch.__version__, "| 장치", DEVICE, "| VIEW", VIEW, "| MASK_STYLE", MASK_STYLE, "| 시드", SEEDS)
if DEVICE == "cpu" and not SMOKE_TEST:
    print("경고: GPU가 없습니다. CPU에서는 미세조정·인코딩이 매우 느립니다 (Colab GPU 런타임 권장)")

In [ ]:
# ======================================================================
# [2] 레시피 텍스트 · 태그
# ======================================================================
csv.field_size_limit(10**9)
for path in (DATA, RAW_RECIPES, PAIRS_CSV) + ((MASKED_RECIPES,) if VIEW == "masked" else ()):
    if not path.exists():
        raise FileNotFoundError(f"{path}가 없습니다")

artifact = torch.load(DATA, map_location="cpu", weights_only=True)
RECIPE_IDS = [str(r) for r in artifact["recipe_ids"]]
SPLITS = {k: [tuple(int(x) for x in row) for row in v] for k, v in artifact["splits"].items()}
if SMOKE_TEST:
    SPLITS = {"train": SPLITS["train"][:96], "val": SPLITS["val"][:48], "test": SPLITS["test"][:48]}
# 평가할 레시피 (스모크 테스트에서는 부분 집합만 인코딩)
NEEDED = sorted({i for rows in SPLITS.values() for a, b, _ in rows for i in (a, b)})

# ---------------- 태그 필터 (stage1_tag_experiments.ipynb와 동일) ----------------
# 모든 레시피에 붙는 상위 분류 (정보 없음)
GENERIC_TAGS = set("""
preparation time-to-make course main-ingredient dietary occasion cuisine equipment
number-of-servings taste-mood low-in-something high-in-something free-of-something
""".split()) | {""}
# 재료를 드러내는 태그: "재료는 달라도 조리법·완성 형태가 비슷한 음식"이 목표이므로 제외
INGREDIENT_TAGS = set("""
meat vegetables fruit pasta-rice-and-grains eggs-dairy poultry chicken beef seafood pasta cheese
pork chocolate beans ground-beef fish potatoes eggs chicken-breasts rice berries nuts
saltwater-fish shellfish tropical-fruit grains apples tomatoes onions squash salmon citrus
shrimp mushrooms greens bananas coconut strawberries peppers broccoli spaghetti elbow-macaroni
pineapple pitted-fruit corn crab asparagus carrots ham bacon cauliflower lentils
long-grain-rice lemon freshwater-fish spinach chicken-thighs-legs pork-sausage blueberries
penne black-beans white-rice pork-loins turkey yams-sweet-potatoes tuna oranges
chick-peas-garbanzos lamb-sheep soy-tofu green-yellow-beans roast-beef raspberries peaches
steaks scallops wings pork-ribs mango lettuces cherries ravioli-tortellini lime pears gelatin
whole-chicken pasta-shells wild-game brown-rice steak beef-ribs bass deer veal catfish
tilapia beef-sausage clams plums melons grapes sole-and-flounder halibut cod turkey-breasts
mussels avocado a1-sauce peanut-butter oysters oatmeal mahi-mahi zucchini chicken-livers
trout papaya beef-organ-meats duck orange-roughy rabbit crawfish medium-grain-rice pheasant
short-grain-rice pickeral beef-liver kiwifruit whitefish duck-breasts bok-choys eggplant
""".split())
# 영양·식단 태그: 대부분 재료 구성으로 정해지므로 제외
DIET_TAGS = set("""
low-sodium healthy low-carb low-calorie low-cholesterol low-protein low-saturated-fat low-fat
healthy-2 vegan vegetarian very-low-carbs high-protein diabetic gluten-free egg-free
high-calcium lactose kosher dairy-free high-fiber nut-free non-alcoholic
""".split())
EXCLUDED_TAGS = GENERIC_TAGS | INGREDIENT_TAGS | DIET_TAGS

EXCLUDED_TAGS = GENERIC_TAGS | INGREDIENT_TAGS | DIET_TAGS
# 재료 이름의 마지막 단어를 따로 가릴 때 제외할 단어 (완성 형태를 나타내는 말)
FORM_WORDS = {"sauce", "dough", "crust", "mix", "soup", "paste", "dressing", "stock", "broth",
              "glaze", "batter", "filling", "frosting", "syrup", "salsa", "dip", "spread"}


def parse_list(value: str) -> list[str]:
    try:
        items = json.loads(value)
    except (json.JSONDecodeError, TypeError):
        items = ast.literal_eval(value)
    return [item if isinstance(item, str) else (item.get("text") or "") for item in items]


def restyle_masks(text: str) -> str:
    # 마스킹 CSV의 [MASK:I0|vegetables|composition_flavor] 형식을 MASK_STYLE에 맞게 바꿈
    if MASK_STYLE == "keep":
        return text

    def replace_token(match):
        parts = match.group(1).split("|")
        if MASK_STYLE == "category" and len(parts) > 1:
            return f"[MASK:{parts[1]}]"
        return "[MASK]"

    return re.sub(r"\[MASK:([^\]]*)\]", replace_token, text)


def mask_ingredients(text: str, ingredients: list[str]) -> str:
    """레시피의 재료 이름을 [MASK]로 바꿈 (레시피 이름, 마스킹 CSV가 없는 레시피에 사용)

    재료 문구 전체, 마지막 단어(예: "baking potatoes" -> "potatoes"), 그리고 문구 안에서
    재료 사전(INGREDIENT_VOCAB)에 있는 부분(예: "chicken drumsticks" -> "chicken")을 가린다.
    """
    names = set()
    for ingredient in ingredients:
        ingredient = ingredient.strip().lower()
        if not ingredient:
            continue
        names.add(ingredient)
        words = ingredient.split()
        if len(words[-1]) >= 4 and words[-1] not in FORM_WORDS:
            names.add(words[-1])
        for size in range(1, len(words)):
            for start in range(len(words) - size + 1):
                part = " ".join(words[start : start + size])
                if part in INGREDIENT_VOCAB:
                    names.add(part)
    for name in sorted(names, key=len, reverse=True):
        text = re.sub(rf"(?<![\w\[]){re.escape(name)}(?:e?s)?(?![\w\]])", "[MASK]", text, flags=re.I)
    return text


def build_text(name: str, steps: list[str], ingredients: list[str], masked_steps: list[str] | None = None) -> str:
    if VIEW == "raw":
        return f"Recipe: {name}. Steps: {' '.join(steps)}"  # training.stage1.recipe_text와 같음
    body = restyle_masks(" ".join(masked_steps)) if masked_steps is not None else mask_ingredients(" ".join(steps), ingredients)
    title = mask_ingredients(name, ingredients) if MASK_NAME else name
    return f"Recipe: {title}. Steps: {body}"


def read_rows(path: Path) -> dict[str, dict]:
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        rows = {}
        for row in csv.DictReader(handle):
            rows.setdefault(row["id"].strip(), row)
        return rows


raw_rows = read_rows(RAW_RECIPES)
# 재료 사전: 라벨링 레시피 재료 목록에 2번 이상 나온 재료 이름 (모든 조건에서 같은 텍스트가 되도록 v3에서만 만듦)
_ingredient_counts = Counter(i.strip().lower() for rid in RECIPE_IDS for i in parse_list(raw_rows[rid]["ingredients"]))
INGREDIENT_VOCAB = {i for i, c in _ingredient_counts.items() if c >= 2 and i not in FORM_WORDS}
masked_rows = read_rows(MASKED_RECIPES) if VIEW == "masked" else {}
missing = [rid for rid in RECIPE_IDS if rid not in raw_rows or (VIEW == "masked" and rid not in masked_rows)]
if missing:
    raise ValueError(f"텍스트가 없는 레시피 {len(missing)}개: {missing[:5]}")
TEXTS = [
    build_text(
        raw_rows[rid]["name"],
        parse_list(raw_rows[rid]["steps"]),
        parse_list(raw_rows[rid]["ingredients"]),
        parse_list(masked_rows[rid]["steps"]) if VIEW == "masked" else None,
    )
    for rid in RECIPE_IDS
]

TAGS: dict[str, list[str]] = {}
with PAIRS_CSV.open(encoding="utf-8-sig", newline="") as handle:
    for row in csv.DictReader(handle):
        for side in ("a", "b"):
            rid = row[f"{side}_id"].strip()
            if rid not in TAGS:
                TAGS[rid] = [t for t in ast.literal_eval(row[f"{side}_tags"]) if t not in EXCLUDED_TAGS]
LABELED_TAGS = [TAGS.get(rid, []) for rid in RECIPE_IDS]

print(f"레시피 {len(RECIPE_IDS)}개 | 쌍 train/val/test = "
      + "/".join(str(len(SPLITS[k])) for k in ("train", "val", "test")))
print("입력 예시:", TEXTS[0][:400], "...")
print("태그 예시:", LABELED_TAGS[0])

_extra_cache: dict[tuple[int, int], tuple[list[str], list[list[str]]]] = {}


def load_extra_recipes(count: int, seed: int) -> tuple[list[str], list[list[str]]]:
    """RAW_recipes_normalized.csv에서 라벨링되지 않은 레시피 count개 (태그 쌍 학습용 텍스트, 태그)"""
    if (count, seed) in _extra_cache:
        return _extra_cache[(count, seed)]
    path = next((p for p in RAW_ALL_CANDIDATES if p.exists()), None)
    if path is None:
        raise FileNotFoundError("RAW_recipes_normalized.csv가 없습니다")
    known = set(RECIPE_IDS)
    pool = []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            rid = row["id"].strip()
            if rid in known or not row.get("steps"):
                continue
            tags = [t for t in ast.literal_eval(row["tags"]) if t not in EXCLUDED_TAGS]
            if tags:
                pool.append((row["name"] or "", row["steps"], row["ingredients"], tags))
    chosen = random.Random(seed).sample(pool, min(count, len(pool)))
    texts = [build_text(name, parse_list(steps), parse_list(ings)) for name, steps, ings, _ in chosen]
    result = (texts, [tags for *_, tags in chosen])
    _extra_cache[(count, seed)] = result
    print(f"RAW에서 레시피 {len(texts)}개 추가 (후보 {len(pool)}개) | 예시: {texts[0][:200]} ...")
    return result

In [ ]:
# ======================================================================
# [3] 손실 · 배치 · 인코딩 · 평가 · 학습 루프
# ======================================================================


def cosent(cos: torch.Tensor, target: torch.Tensor, scale: float, margin: float = 0.0):
    """CoSENT: target_i > target_j + margin인 모든 (i, j)에 대해 cos_i > cos_j가 되도록 하는 순서 손실"""
    score = cos * scale
    diff = score[None, :] - score[:, None]  # [i, j] = s_j - s_i
    keep = target[:, None] > target[None, :] + margin
    if not keep.any():
        return None
    return torch.logsumexp(torch.cat((diff.new_zeros(1), diff[keep])), dim=0)


def label_batches(pairs, rng: random.Random, pairs_per_batch: int):
    """같은 기준 레시피(쌍이 더 많은 쪽)의 쌍을 한 배치에 묶음"""
    count = Counter(i for a, b, _ in pairs for i in (a, b))
    groups = defaultdict(list)
    for a, b, s in pairs:
        groups[a if count[a] >= count[b] else b].append((a, b, float(s)))
    ordered = list(groups.values())
    rng.shuffle(ordered)
    batches, current = [], []
    for group in ordered:
        group = group[:]
        rng.shuffle(group)
        while group:
            space = pairs_per_batch - len(current)
            current, group = current + group[:space], group[space:]
            if len(current) >= pairs_per_batch:
                batches.append(current)
                current = []
    if current:
        batches.append(current)
    return batches


class TagPool:
    """태그 IDF 가중 Jaccard (목표값)와 태그 IDF 코사인 상위 이웃 (샘플링용)"""

    def __init__(self, tag_lists: list[list[str]], near_k: int = 50):
        self.n = len(tag_lists)
        self.sets = [set(tags) for tags in tag_lists]
        df = Counter(t for tags in self.sets for t in tags)
        self.idf = {t: math.log(self.n / c) for t, c in df.items()}
        vocab = {t: i for i, t in enumerate(sorted(df))}
        matrix = torch.zeros(self.n, len(vocab))
        for row, tags in enumerate(self.sets):
            for t in tags:
                matrix[row, vocab[t]] = self.idf[t]
        matrix = F.normalize(matrix, dim=1).to(DEVICE)
        near = []
        k = min(near_k, self.n - 1)
        for start in range(0, self.n, 2048):
            sim = matrix[start : start + 2048] @ matrix.T
            rows = torch.arange(sim.shape[0], device=DEVICE)
            sim[rows, rows + start] = -1  # 자기 자신 제외
            near.append(sim.topk(k, dim=1).indices.cpu())
        self.near = torch.cat(near).tolist()

    def similarity(self, i: int, j: int) -> float:
        union = self.sets[i] | self.sets[j]
        total = sum(self.idf[t] for t in union)
        return sum(self.idf[t] for t in self.sets[i] & self.sets[j]) / total if total else 0.0

    def batch(self, rng: random.Random, anchors: int, banned: set, near_per: int = 2, random_per: int = 2):
        pairs = []
        for a in [rng.randrange(self.n) for _ in range(anchors)]:
            partners = rng.sample(self.near[a], near_per) + [rng.randrange(self.n) for _ in range(random_per)]
            for b in partners:
                if b != a and (min(a, b), max(a, b)) not in banned:
                    pairs.append((a, b, self.similarity(a, b)))
        return pairs


def embed(model, texts: list[str]) -> torch.Tensor:
    # sentence-transformers 버전에 따라 tokenize 결과에 텐서가 아닌 값도 들어 있음
    features = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in model.tokenize(texts).items()}
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
        output = model(features)["sentence_embedding"]
    return F.normalize(output.float(), dim=-1)


def pair_cosines(model, pairs, texts: list[str]) -> torch.Tensor:
    nodes = sorted({i for a, b, _ in pairs for i in (a, b)})
    position = {n: k for k, n in enumerate(nodes)}
    vectors = embed(model, [texts[n] for n in nodes])
    left = torch.tensor([position[a] for a, _, _ in pairs], device=DEVICE)
    right = torch.tensor([position[b] for _, b, _ in pairs], device=DEVICE)
    return (vectors[left] * vectors[right]).sum(-1)


def token_windows(text: str, tokenizer, limit: int) -> list[tuple[str, int]]:
    """src.engine._token_windows와 동일 (ordered_token_windows_v1): 잘라내지 않고 연속된 창으로 나눔"""
    offsets = tokenizer(text, add_special_tokens=False, truncation=False,
                        return_offsets_mapping=True, verbose=False)["offset_mapping"]
    if not offsets:
        return [(text, 1)]
    budget = limit - tokenizer.num_special_tokens_to_add(pair=False) - 8
    windows, start = [], 0
    while start < len(offsets):
        end = min(start + budget, len(offsets))
        while True:
            begin_char = 0 if start == 0 else offsets[start][0]
            end_char = len(text) if end == len(offsets) else offsets[end][0]
            window = text[begin_char:end_char]
            if len(tokenizer(window, add_special_tokens=True, truncation=False, verbose=False)["input_ids"]) <= limit:
                break
            end -= 1
            if end <= start:
                raise ValueError("could not fit a token into the encoder window")
        windows.append((window, end - start))
        start = end
    return windows


_windows = {}


@torch.no_grad()
def encode_recipes(model) -> torch.Tensor:
    """NEEDED 레시피를 기존 1단계와 같은 방식(창별 임베딩을 토큰 수로 가중 평균 후 L2 정규화)으로 인코딩"""
    model.eval()
    if EVAL_LIMIT not in _windows:
        _windows[EVAL_LIMIT] = [token_windows(TEXTS[i], model.tokenizer, EVAL_LIMIT) for i in NEEDED]
    windows = _windows[EVAL_LIMIT]
    saved_limit, model.max_seq_length = model.max_seq_length, EVAL_LIMIT
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
        vectors = model.encode([w for parts in windows for w, _ in parts], batch_size=64,
                               convert_to_tensor=True, normalize_embeddings=True).float().cpu()
    model.max_seq_length = saved_limit
    features = torch.zeros(len(TEXTS), vectors.shape[1])
    cursor = 0
    for index, parts in zip(NEEDED, windows):
        weights = torch.tensor([count for _, count in parts], dtype=torch.float32)
        features[index] = F.normalize((vectors[cursor : cursor + len(parts)] * weights[:, None]).sum(0), dim=0)
        cursor += len(parts)
    return features


def concordance(values, labels, low: int, high: int) -> float:
    # 라벨 high 쌍의 코사인이 라벨 low 쌍보다 큰 비율 (동점 0.5)
    a, b = values[labels == low], values[labels == high]
    if not len(a) or not len(b):
        return float("nan")
    return float((b[:, None] > a[None]).mean() + 0.5 * (b[:, None] == a[None]).mean())


def fit_thresholds(values, labels) -> np.ndarray:
    # 정확도가 최대가 되는 4개 경계 (좌표 탐색)
    prior = np.cumsum(np.bincount(labels, minlength=5))[:4] / len(labels)
    thresholds = np.quantile(values, prior)
    grid = np.quantile(values, np.linspace(0.005, 0.995, 199))
    for _ in range(4):
        for j in range(4):
            def accuracy(q):
                trial = np.sort(np.r_[thresholds[:j], q, thresholds[j + 1 :]])
                return (np.searchsorted(trial, values, side="right") == labels).mean()
            thresholds[j] = max(grid, key=accuracy)
            thresholds = np.sort(thresholds)
    return thresholds


def evaluate(features: torch.Tensor) -> dict:
    """쌍 코사인 기준 지표. 경계는 검증에서 맞춰 모든 분할에 적용 (테스트 정확도가 정직한 값)"""
    cos, labels = {}, {}
    for part, rows in SPLITS.items():
        rows_t = torch.tensor(rows)
        cos[part] = (features[rows_t[:, 0]] * features[rows_t[:, 1]]).sum(1).numpy()
        labels[part] = rows_t[:, 2].numpy()
    thresholds = fit_thresholds(cos["val"], labels["val"])
    report = {"thresholds_from_val": thresholds.tolist()}
    for part in ("train", "val", "test"):
        c, y = cos[part], labels[part]
        predicted = np.searchsorted(thresholds, c, side="right")
        matrix = [[int(((y == a) & (predicted == p)).sum()) for p in range(5)] for a in range(5)]
        report[part] = {
            "n": len(y),
            "spearman": float(spearmanr(c, y)[0]),
            "concordance_1_2": concordance(c, y, 1, 2),
            "concordance_2_3": concordance(c, y, 2, 3),
            "concordance_0_4": concordance(c, y, 0, 4),
            "exact_accuracy": float((predicted == y).mean()),
            "within_one_accuracy": float((np.abs(predicted - y) <= 1).mean()),
            "per_class_recall": [matrix[k][k] / sum(matrix[k]) if sum(matrix[k]) else float("nan") for k in range(5)],
            "confusion_matrix": matrix,
            "cosine_mean_by_class": [float(c[y == k].mean()) if (y == k).any() else float("nan") for k in range(5)],
        }
    return report


@dataclass(frozen=True)
class Config:
    label_weight: float = 1.0  # 라벨 CoSENT 손실 가중치 (0이면 사용 안 함)
    tag_weight: float = 0.0  # 태그 IDF Jaccard CoSENT 손실 가중치
    extra_recipes: int = 0  # 태그 쌍 풀에 추가할 RAW 레시피 수
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup: float = 0.1  # 전체 스텝 중 학습률 워밍업 비율
    max_epochs: int = 6
    patience: int = 2  # 검증 Spearman이 이 epoch 수만큼 안 오르면 중단
    pairs_per_batch: int = 16  # 라벨 배치당 쌍 수
    tag_anchors: int = 4  # 태그 배치: 기준 레시피 수 × (가까운 2 + 무작위 2)
    tag_margin: float = 0.05
    scale: float = 20.0  # CoSENT 온도 (코사인 × scale)
    freeze_layers: int = 6  # 임베딩 + 아래 BERT 층 몇 개를 고정할지 (과적합 방지)
    train_max_seq: int = 512  # 학습 시 입력 길이 (넘으면 앞부분만 사용)
    view: str = VIEW
    mask_style: str = MASK_STYLE
    mask_name: bool = MASK_NAME
    seed: int = 42
    smoke_test: bool = SMOKE_TEST


def load_model(config: Config):
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    model.float()  # 체크포인트가 fp16이어도 fp32 가중치로 학습 (autocast로 연산만 fp16)
    model.max_seq_length = config.train_max_seq
    bert = model[0].auto_model
    frozen = [bert.embeddings] + list(bert.encoder.layer[: config.freeze_layers])
    for module in frozen:
        for parameter in module.parameters():
            parameter.requires_grad_(False)
    return model


def summary_line(epoch, report, loss=None):
    v, t = report["val"], report["test"]
    loss_text = f" loss={loss:.4f}" if loss is not None else ""
    return (f"epoch {epoch}{loss_text} | train ρ={report['train']['spearman']:.3f} | val ρ={v['spearman']:.3f} "
            f"순서 1<2 {v['concordance_1_2']:.2f} 2<3 {v['concordance_2_3']:.2f} | "
            f"test ρ={t['spearman']:.3f} 정확 {t['exact_accuracy']:.1%}")


def run(label: str, config: Config) -> dict:
    out = RESULTS / label / f"seed{config.seed}"
    path = out / "metrics.json"
    if path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))
        if saved["settings"] == asdict(config):
            print(f"[{label} seed={config.seed}] 기존 결과 사용")
            return saved
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    rng = random.Random(config.seed)
    started = time.time()
    model = load_model(config)

    texts = TEXTS
    pool = None
    if config.tag_weight:
        tag_lists = LABELED_TAGS
        if config.extra_recipes:
            extra_texts, extra_tags = load_extra_recipes(config.extra_recipes, config.seed)
            texts, tag_lists = TEXTS + extra_texts, LABELED_TAGS + extra_tags
        pool = TagPool(tag_lists)
    banned = {(min(a, b), max(a, b)) for part in ("val", "test") for a, b, _ in SPLITS[part]}

    report = evaluate(encode_recipes(model))
    history = [{"epoch": 0, "val": report["val"], "test": report["test"], "train_spearman": report["train"]["spearman"]}]
    print(f"[{label} seed={config.seed}] " + summary_line(0, report))
    best_score, best_epoch, best_state, stale = report["val"]["spearman"], 0, None, 0

    if config.label_weight or config.tag_weight:
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=config.lr, weight_decay=config.weight_decay)
        # 라벨·태그 조건 모두 epoch당 스텝 수를 라벨 배치 수에 맞춤 (계산량 동일)
        steps_per_epoch = len(label_batches(SPLITS["train"], random.Random(0), config.pairs_per_batch))
        total = steps_per_epoch * config.max_epochs
        warmup = max(1, int(total * config.warmup))
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lambda step: min((step + 1) / warmup, max(0.0, (total - step) / max(1, total - warmup)))
        )
        scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
        for epoch in range(1, config.max_epochs + 1):
            model.train()
            batches = label_batches(SPLITS["train"], rng, config.pairs_per_batch)[:steps_per_epoch]
            losses = []
            for batch in batches:
                terms = []
                if config.label_weight:
                    cos = pair_cosines(model, batch, texts)
                    loss = cosent(cos, torch.tensor([s for *_, s in batch], device=DEVICE), config.scale)
                    if loss is not None:
                        terms.append(config.label_weight * loss)
                if config.tag_weight:
                    tag_batch = pool.batch(rng, config.tag_anchors, banned)
                    if tag_batch:
                        cos = pair_cosines(model, tag_batch, texts)
                        target = torch.tensor([s for *_, s in tag_batch], device=DEVICE)
                        loss = cosent(cos, target, config.scale, config.tag_margin)
                        if loss is not None:
                            terms.append(config.tag_weight * loss)
                if not terms:
                    continue
                loss = sum(terms)
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                losses.append(float(loss))
            report = evaluate(encode_recipes(model))
            mean_loss = statistics.fmean(losses) if losses else float("nan")
            history.append({"epoch": epoch, "train_loss": mean_loss, "val": report["val"], "test": report["test"],
                            "train_spearman": report["train"]["spearman"]})
            print(f"[{label} seed={config.seed}] " + summary_line(epoch, report, mean_loss), flush=True)
            if report["val"]["spearman"] > best_score:
                best_score, best_epoch, stale = report["val"]["spearman"], epoch, 0
                best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
            else:
                stale += 1
                if stale >= config.patience:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        else:  # 한 번도 기준선보다 좋아지지 않음: 미세조정 전 가중치로 되돌림
            model = load_model(config)

    features = encode_recipes(model)
    report = evaluate(features)
    result = {"settings": asdict(config), "best_epoch": best_epoch, "epochs_run": len(history) - 1,
              "minutes": (time.time() - started) / 60, **{k: report[k] for k in ("train", "val", "test")},
              "thresholds_from_val": report["thresholds_from_val"]}
    out.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(result, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    (out / "history.json").write_text(json.dumps(history, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    if not SMOKE_TEST:
        # 기존 1단계 pt와 같은 형식: stage1_graph_experiments.ipynb의 DATA로 바로 사용 가능
        torch.save({
            "recipe_ids": artifact["recipe_ids"], "recipe_features": features, "splits": artifact["splits"],
            "seed": artifact.get("seed", 42), "encoder": f"{MODEL_NAME}+cosent:{label}",
            "view": f"{VIEW}({MASK_STYLE}, mask_name={MASK_NAME})", "encoding_strategy": "ordered_token_windows_v1",
            "finetune_settings": asdict(config),
        }, out / "stage1_embeddings.pt")
    print(f"[{label} seed={config.seed}] best epoch {best_epoch} | {result['minutes']:.1f}분 | 저장: {out}")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return result

## 실험

조건은 `CONDITIONS`에서 고친다. `+extra` 조건은 `RAW_recipes_normalized.csv`가 없으면 건너뛴다. 결과가 이미 있는 조건·시드는 다시 학습하지 않는다.

In [ ]:
# ======================================================================
# [4] 실험 실행
# ======================================================================
BASE = Config()
CONDITIONS = {
    "frozen": {"label_weight": 0.0, "tag_weight": 0.0},
    "label": {"label_weight": 1.0},
    "tag": {"label_weight": 0.0, "tag_weight": 1.0},
    "label+tag0.2": {"label_weight": 1.0, "tag_weight": 0.2},
    "tag+extra": {"label_weight": 0.0, "tag_weight": 1.0, "extra_recipes": 20000},
    "label+tag0.2+extra": {"label_weight": 1.0, "tag_weight": 0.2, "extra_recipes": 20000},
}
if SMOKE_TEST:
    BASE = replace(BASE, max_epochs=1, pairs_per_batch=8, tag_anchors=2, train_max_seq=64, freeze_layers=11)
    CONDITIONS = {k: {**v, "extra_recipes": min(v.get("extra_recipes", 0), 50)} for k, v in CONDITIONS.items()}

has_raw_all = any(p.exists() for p in RAW_ALL_CANDIDATES)
results = {}
for label, overrides in CONDITIONS.items():
    if overrides.get("extra_recipes") and not has_raw_all:
        print(f"[{label}] RAW_recipes_normalized.csv가 없어 건너뜀")
        continue
    for seed in SEEDS:
        results[(label, seed)] = run(label, replace(BASE, **overrides, seed=seed))

In [ ]:
# ======================================================================
# [5] 요약 (시드 평균 ± 표준편차, 검증 Spearman 순)
# ======================================================================
from IPython.display import Markdown, display

METRICS = (
    ("val", "spearman", "검증 ρ", 3), ("val", "concordance_1_2", "검증 순서 1<2", 2),
    ("val", "concordance_2_3", "검증 순서 2<3", 2), ("test", "spearman", "테스트 ρ", 3),
    ("test", "exact_accuracy", "테스트 정확", 3), ("test", "within_one_accuracy", "테스트 ±1", 3),
    ("train", "spearman", "학습 ρ (과적합 확인)", 3),
)


def mean_std(values, digits):
    if len(values) == 1:
        return f"{values[0]:.{digits}f}"
    return f"{statistics.fmean(values):.{digits}f} ± {statistics.stdev(values):.{digits}f}"


rows = []
for label in CONDITIONS:
    runs = [r for (name, _), r in results.items() if name == label]
    if runs:
        rows.append((label, runs))
rows.sort(key=lambda item: -statistics.fmean(r["val"]["spearman"] for r in item[1]))
header = "| 순위 | 조건 | " + " | ".join(m[2] for m in METRICS) + " | best epoch | 클래스별 재현율 0~4 (테스트) |"
lines = [header, "|" + "---|" * (len(METRICS) + 4)]
csv_rows = []
for rank, (label, runs) in enumerate(rows, 1):
    cells = [mean_std([r[part][key] for r in runs], d) for part, key, _, d in METRICS]
    recall = np.nanmean([r["test"]["per_class_recall"] for r in runs], axis=0)
    epochs = mean_std([r["best_epoch"] for r in runs], 1)
    lines.append(f"| {rank} | {label} | " + " | ".join(cells) + f" | {epochs} | " + " / ".join(f"{x:.2f}" for x in recall) + " |")
    csv_rows.append({"condition": label, "runs": len(runs),
                     **{f"{p}_{k}_mean": statistics.fmean(r[p][k] for r in runs) for p, k, _, _ in METRICS},
                     "best_epoch_mean": statistics.fmean(r["best_epoch"] for r in runs)})
table = "\n".join(lines)
display(Markdown(table))
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "summary.md").write_text(table + "\n", encoding="utf-8")
with (RESULTS / "summary.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(csv_rows[0]))
    writer.writeheader()
    writer.writerows(csv_rows)
print("참고: 고정 GTE 원본 입력(VIEW='raw', 기존 1단계)의 쌍 코사인 Spearman은 0.617, 순서 일치 1<2 0.57 · 2<3 0.63")

## GraphSAGE에 붙여보기

검증 Spearman이 `frozen`보다 확실히 오른 조건만 GraphSAGE에 붙인다.

1. `results/experiments/05_encoder_finetune/<조건>/seed<시드>/stage1_embeddings.pt`를 `stage1_graph_experiments.ipynb`의 `DATA`로 지정하고 실험 1 셀(또는 `Config()` 한 조건)을 실행한다. 레시피 순서와 분할이 기존 pt와 같아서 그대로 쓸 수 있다.
2. **주의:** 인코더가 학습 쌍의 코사인 순서를 이미 맞춰 두었기 때문에, 같은 학습 쌍으로 GraphSAGE를 학습하면 코사인 특징(`raw_cosine_feature`)을 지나치게 믿게 될 수 있다. 검증·테스트 차이를 꼭 같이 보고, `raw_cosine_feature=False` 조건도 비교한다.

In [ ]:
# ======================================================================
# 결과 내려받기 (Colab 전용): 임베딩 pt 포함
# ======================================================================
import shutil

archive = shutil.make_archive("stage1_encoder_finetune", "zip", RESULTS)
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print(f"Colab이 아니므로 다운로드 대신 파일만 만듦: {archive}")